In [1]:
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import MessagesState
from langchain_deepseek import ChatDeepSeek
from rich import print as rprint

from dotenv import load_dotenv
load_dotenv(override=True)

#0. 连接LLM模型
model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={
        "thinking":{
            "type":"disabled"
        }
    }
)

#1. 定义状态
class OverAllState(MessagesState):
    username: str
    output: str

#2. 定义节点
def node_a(state: OverAllState) -> OverAllState:
    return {
        "messages": [HumanMessage("你好,我是" + state["username"])],
    }

def llm_node(state: OverAllState) -> OverAllState:
    res = model.invoke(state["messages"])
    return {
        "messages": [res],
        "output": res.content
    }

#3. 构建图
builder = StateGraph(state_schema=OverAllState)
builder.add_node("node_a", node_a)
builder.add_node("llm_node", llm_node)
builder.add_edge(START, "node_a")
builder.add_edge("node_a", "llm_node")
builder.add_edge("llm_node", END)

graph = builder.compile()

#4. 运行图
result = graph.invoke({"username": "老王"})
rprint(result)


ValidationError: 1 validation error for ChatDeepSeek
  Value error, If using default api base, DEEPSEEK_API_KEY must be set. [type=value_error, input_value={'model': 'deepseek-v4-fl...'}}, 'model_kwargs': {}}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error